In [1]:
!pip install openmeteo-requests retry-requests requests-cache

  Using cached retry_requests-2.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
Using cached retry_requests-2.0.0-py3-none-any.whl (15 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
   ---------------------------------------- 0.0/758.1 kB ? eta -:--:--
   ---------------------------------------- 758.1/758.1 kB 12.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 12.5 MB/s  0:00:00
Using cached flatbuffers-25.9.23-py2.py3-none-any.whl (30 kB)

   ------ ---------------------------------  2/13 [url-normalize]
   ------ ---------------------------------  2/13 [url-normalize]
   --------- ------------------------------  3/13 [qh3]
   ------------ ---------------------------  4/13 [openmeteo-sdk]
   --------------- ------------------------  5/13 [jh2]
  Attempting u

In [2]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta

def setup_client():
    """
    Menyiapkan client API dengan fitur Cache dan Retry.
    Tujuannya agar koneksi stabil dan hemat kuota.
    """
    # Buat cache bernama '.cache', data disimpan selama 3600 detik (1 jam)
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # Jika gagal connect, coba lagi (retry) sampai 5x
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # Return objek client yang siap pakai
    return openmeteo_requests.Client(session=retry_session)

In [3]:
# ==========================================
# 2. FETCH DATA PER CHUNK
# ==========================================
def fetch_chunk(client, lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", 
                   "relative_humidity_2m", 
                   "dew_point_2m", 
                   "rain", 
                   "wind_speed_10m",
                   "wind_gusts_10m",
                   "wind_direction_10m", 
                   "surface_pressure",
                   "pressure_msl",
                   "sunshine_duration",
                   "direct_radiation",
                   "cloud_cover"],
        "models": "era5",
        "timezone": "Asia/Bangkok"
    }

    print(f"   ⏳ Mengambil: {start_date} s.d {end_date}...")
    
    # --- LOGIKA RETRY / ANTI-LIMIT ---
    max_retries = 5
    attempt = 0
    
    while attempt < max_retries:
        try:
            # Coba panggil API
            responses = client.weather_api(url, params=params)
            
            # Jika berhasil, langsung proses dan kembalikan data
            return process_data(responses[0]) 

        except Exception as e:
            error_msg = str(e)
            
            # CEK APAKAH ERROR KARENA LIMIT?
            if "Minutely API request limit exceeded" in error_msg or "429" in error_msg:
                print(f"      ⛔ Kena Limit API! Menunggu 70 detik sebelum coba lagi... (Percobaan {attempt+1}/{max_retries})")
                # Tunggu 60 detik + 10 detik buffer biar aman
                time.sleep(70)
                attempt += 1
            else:
                # Jika error lain (misal koneksi putus total), print dan coba lagi sebentar
                print(f"      ❌ Error lain: {e}. Menunggu 5 detik...")
                time.sleep(5)
                attempt += 1
    
    # Jika sudah mencoba 5x masih gagal juga
    print(f"   💀 Gagal total mengambil chunk {start_date} setelah {max_retries} kali percobaan.")
    return None

# ==========================================
# 3. PROCESS DATA (UBAHAN: Date Jadi Kolom Biasa)
# ==========================================
def process_data(response):
    hourly = response.Hourly()

    date_range = pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    df = pd.DataFrame(data = {
        "date": date_range, # Date tetap jadi kolom biasa
        "temperature_era5": hourly.Variables(0).ValuesAsNumpy(),
        "humidity_era5": hourly.Variables(1).ValuesAsNumpy(),
        "dewpoint_era5":hourly.Variables(2).ValuesAsNumpy(),
        "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed_era5": hourly.Variables(4).ValuesAsNumpy(),
        "wind_gusts_era5": hourly.Variables(5).ValuesAsNumpy(),
        "wind_direction_era5": hourly.Variables(6).ValuesAsNumpy(),
        "pressure_era5": hourly.Variables(7).ValuesAsNumpy(),
        "sealevel_pressure_era5": hourly.Variables(8).ValuesAsNumpy(),
        "sunshine_duration_era5" : hourly.Variables(9).ValuesAsNumpy(),
        "direct_radiation_era5" : hourly.Variables(10).ValuesAsNumpy(),
        "cloud_cover_era5": hourly.Variables(11).ValuesAsNumpy(),
    })

    # Konversi timezone kolom date (Tanpa set_index)
    df['date'] = df['date'].dt.tz_convert('Asia/Jakarta')
    
    return df

# ==========================================
# 4. LOGIKA UTAMA: CHUNKING & MERGING (SAFE MODE)
# ==========================================
def fetch_long_period_data(client, lat, lon, start_str, end_str, folder_tujuan, chunk_years=10):
    
    # ... (Bagian awal sama) ...
    if not os.path.exists(folder_tujuan):
        os.makedirs(folder_tujuan)

    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    final_end_date = datetime.strptime(end_str, "%Y-%m-%d")
    all_files = []
    current_start = start_date

    print(f"🚀 Memulai Misi Pengambilan Data...")

    # --- FASE 1: DOWNLOAD ---
    while current_start < final_end_date:
        current_end = current_start.replace(year=current_start.year + chunk_years) - timedelta(days=1)
        if current_end > final_end_date:
            current_end = final_end_date

        s_str = current_start.strftime("%Y-%m-%d")
        e_str = current_end.strftime("%Y-%m-%d")
        chunk_filename = os.path.join(folder_tujuan, f"data_{s_str}_{e_str}.csv")

        if os.path.exists(chunk_filename):
            print(f"⏩ Skip: {s_str} - {e_str} (Sudah ada)")
            all_files.append(chunk_filename)
        else:
            # Panggil fetch_chunk yang SUDAH DIMODIFIKASI di atas
            df_chunk = fetch_chunk(client, lat, lon, s_str, e_str)
            
            if df_chunk is not None:
                df_chunk.to_csv(chunk_filename, index=False)
                print(f"   ✅ Tersimpan: {chunk_filename}")
                all_files.append(chunk_filename)
                
                # Istirahat normal antar chunk (bukan saat error)
                time.sleep(2) 
            else:
                print("   ⚠️ Chunk ini dilewati karena gagal total.")

        current_start = current_end + timedelta(days=1)

    print("\n🔗 Menggabungkan semua pecahan data...")

    # --- FASE 2: MERGE (Sama seperti kode sukses sebelumnya) ---
    df_list = []
    for f in all_files:
        try:
            df = pd.read_csv(f, parse_dates=['date'])
            df_list.append(df)
        except Exception as e:
            print(f"Warning: Gagal membaca file {f}, dilewati.")

    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)
        df_final = df_final.drop_duplicates(subset=['date'], keep='first')
        df_final = df_final.set_index('date')
        df_final = df_final.sort_index()

        print(f"🎉 SUKSES! Total Data: {len(df_final)} baris.")
        return df_final
    else:
        return None

In [4]:
import time
from datetime import date, timedelta
import os
# Pastikan library lain yang dibutuhkan (openmeteo_requests, pandas, retry, dll) sudah terimport di bagian atas filemu

# ==========================================
# FUNGSI TAMBAHAN: RETRY LOGIC (ANTI GAGAL)
# ==========================================
def fetch_chunk_safe(func_fetch, *args, max_retries=5, **kwargs):
    """
    Mencoba menjalankan fungsi fetch. Jika gagal (kena limit API),
    akan menunggu dan mencoba lagi secara otomatis.
    """
    wait_seconds = 60 # Tunggu 1 menit jika gagal (sesuai aturan Open Meteo)

    for attempt in range(1, max_retries + 1):
        try:
            return func_fetch(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ Percobaan ke-{attempt} gagal: {e}")
            if attempt < max_retries:
                print(f"⏳ Menunggu {wait_seconds} detik sebelum mencoba lagi...")
                time.sleep(wait_seconds)
            else:
                print("❌ Gagal total setelah beberapa kali percobaan.")
                raise e # Lempar error jika sudah menyerah

# ==========================================
# EKSEKUSI UTAMA
# ==========================================
if __name__ == "__main__":
    # Konfigurasi
    LAT = -7.736663290223948, 
    LON = 109.64605763039752
    MULAI = "1975-01-01"
    
    # Set Akhir = Kemarin
    kemarin = date.today() - timedelta(days=1)
    AKHIR = kemarin.strftime("%Y-%m-%d")

    FOLDER = "open_meteo_jerukagung"
    FILE_FINAL = "cuaca_jerukagung.csv"

    client = setup_client()

    # Jalankan
    try:
        # PENTING: Gunakan chunk_years lebih kecil (misal 5) agar tidak terlalu berat
        # dan mengurangi risiko timeout di sisi server
        df_lengkap = fetch_long_period_data(client, LAT, LON, MULAI, AKHIR, FOLDER, chunk_years=5)

        if df_lengkap is not None:
            path_final = os.path.join(FOLDER, FILE_FINAL)
            # Simpan final
            df_lengkap.to_csv(path_final) 
            print(f"💾 File Gabungan Tersimpan: {path_final}")
            
    except Exception as e:
        print(f"❌ Terjadi Error Fatal: {e}")

🚀 Memulai Misi Pengambilan Data...
   ⏳ Mengambil: 1975-01-01 s.d 1979-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_1975-01-01_1979-12-31.csv
   ⏳ Mengambil: 1980-01-01 s.d 1984-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_1980-01-01_1984-12-31.csv
   ⏳ Mengambil: 1985-01-01 s.d 1989-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_1985-01-01_1989-12-31.csv
   ⏳ Mengambil: 1990-01-01 s.d 1994-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_1990-01-01_1994-12-31.csv
   ⏳ Mengambil: 1995-01-01 s.d 1999-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_1995-01-01_1999-12-31.csv
   ⏳ Mengambil: 2000-01-01 s.d 2004-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_2000-01-01_2004-12-31.csv
   ⏳ Mengambil: 2005-01-01 s.d 2009-12-31...
   ✅ Tersimpan: open_meteo_jerukagung\data_2005-01-01_2009-12-31.csv
   ⏳ Mengambil: 2010-01-01 s.d 2014-12-31...
      ⛔ Kena Limit API! Menunggu 70 detik sebelum coba lagi... (Percobaan 1/5)
   ✅ Tersimpan: open_meteo_jerukagung\data_